# Experiment 2: Bag of Words vs TF-IDF Comparison with MLflow

Comparing multiple algorithms (Logistic Regression, MultinomialNB, XGBoost, Random Forest, Gradient Boosting) using CountVectorizer and TfidfVectorizer, logging all parent and nested child runs to MLflow & DagsHub.

In [ ]:
import os
import re
import string
import warnings
import pandas as pd
import numpy as np
import scipy.sparse

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import mlflow
import mlflow.sklearn
import dagshub

warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore")

## 1. Setup MLflow Tracking & DagsHub Integration

In [ ]:
DATA_PATH = '../data/balanced_sentiment_dataset.csv' if os.path.exists('../data/balanced_sentiment_dataset.csv') else 'data/balanced_sentiment_dataset.csv'

CONFIG = {
    "data_path": DATA_PATH,
    "test_size": 0.2,
    "mlflow_tracking_uri": os.getenv("MLFLOW_TRACKING_URI", "https://dagshub.com/rohitkr8527/sentiment-analysis.mlflow"),
    "dagshub_repo_owner": os.getenv("DAGSHUB_REPO_OWNER", "rohitkr8527"),
    "dagshub_repo_name": os.getenv("DAGSHUB_REPO_NAME", "sentiment-analysis"),
    "experiment_name": "Bow vs TfIdf"
}

# Connect MLflow and DagsHub
mlflow.set_tracking_uri(CONFIG["mlflow_tracking_uri"])
dagshub_token = os.getenv("DAGSHUB_TOKEN") or os.getenv("sentiment_analysis", "")
if dagshub_token:
    os.environ["MLFLOW_TRACKING_USERNAME"] = dagshub_token
    os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token
    dagshub.init(repo_owner=CONFIG["dagshub_repo_owner"], repo_name=CONFIG["dagshub_repo_name"], mlflow=True)

mlflow.set_experiment(CONFIG["experiment_name"])
print(f"Experiment: '{CONFIG['experiment_name']}' initialized at {CONFIG['mlflow_tracking_uri']}")

## 2. Text Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = ''.join([char for char in text if not char.isdigit()])
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    text = text.replace('؛', '')
    text = re.sub(r'\s+', ' ', text).strip()
    words = [lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    return " ".join(words)

df = pd.read_csv(CONFIG["data_path"])
df['text'] = df['text'].apply(preprocess_text)
print(f"Loaded and preprocessed {len(df)} records.")

## 3. Define Algorithms & Vectorizers

In [ ]:
VECTORIZERS = {
    'BoW': CountVectorizer(max_features=5000),
    'TF-IDF': TfidfVectorizer(max_features=5000)
}

ALGORITHMS = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'MultinomialNB': MultinomialNB(),
    'XGBoost': XGBClassifier(eval_metric='logloss'),
    'RandomForest': RandomForestClassifier(random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42)
}

def log_model_params(algo_name, model):
    params_to_log = {}
    if algo_name == 'LogisticRegression':
        params_to_log["C"] = model.C
    elif algo_name == 'MultinomialNB':
        params_to_log["alpha"] = model.alpha
    elif algo_name == 'XGBoost':
        p = model.get_params()
        params_to_log["n_estimators"] = p.get("n_estimators")
        params_to_log["learning_rate"] = p.get("learning_rate")
    elif algo_name in ['RandomForest', 'GradientBoosting']:
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["max_depth"] = model.max_depth
    mlflow.log_params(params_to_log)

## 4. Execute Multi-Model Comparison with MLflow Logging

In [ ]:
results = []

with mlflow.start_run(run_name="All Experiments Comparison") as parent_run:
    for algo_name, algorithm in ALGORITHMS.items():
        for vec_name, vectorizer in VECTORIZERS.items():
            with mlflow.start_run(run_name=f"{algo_name} with {vec_name}", nested=True) as child_run:
                try:
                    # 1. Vectorize
                    X = vectorizer.fit_transform(df['text'])
                    y = df['sentiment']
                    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=CONFIG["test_size"], random_state=42)

                    # 2. Log Configuration
                    mlflow.log_params({
                        "vectorizer": vec_name,
                        "algorithm": algo_name,
                        "test_size": CONFIG["test_size"],
                        "max_features": 5000
                    })

                    # 3. Train
                    model = algorithm
                    model.fit(X_train, y_train)
                    log_model_params(algo_name, model)

                    # 4. Evaluate & Log Metrics
                    y_pred = model.predict(X_test)
                    metrics = {
                        "accuracy": float(accuracy_score(y_test, y_pred)),
                        "precision": float(precision_score(y_test, y_pred)),
                        "recall": float(recall_score(y_test, y_pred)),
                        "f1_score": float(f1_score(y_test, y_pred))
                    }
                    mlflow.log_metrics(metrics)

                    # 5. Log Model Artifact
                    input_example = X_test[:5].toarray() if scipy.sparse.issparse(X_test) else X_test[:5]
                    mlflow.sklearn.log_model(model, "model", input_example=input_example)

                    results.append({
                        "Algorithm": algo_name,
                        "Vectorizer": vec_name,
                        **metrics
                    })
                    print(f"Logged: {algo_name} + {vec_name} -> Accuracy: {metrics['accuracy']:.4f}, F1: {metrics['f1_score']:.4f}")

                except Exception as e:
                    print(f"Error in {algo_name} with {vec_name}: {e}")
                    mlflow.log_param("error", str(e))

summary_df = pd.DataFrame(results)
print("\n--- Experiment Comparison Summary ---")
display(summary_df) if 'display' in globals() else print(summary_df)